# Chapter 2: From Model to Money — A Strategist's Guide to Real-World Financial AI
**Financial AI in Practice** | Manning Publications

Chapter 2 argues that a financial AI project survives only if it is tied to business value, framed as either an
*offense* play (new revenue) or a *defense* play (efficiency and retention), and it follows the fictional AlphaStream
brokerage as it plans an AI-powered "Alpha Digest." This notebook holds the chapter's two code listings, which isolate
the prompt-engineering step of each strategy: listing 2.1 asks an LLM for a premium, personalized digest of the posts
a user follows (offense), and listing 2.2 repurposes the same technology as an internal moderation assistant that
returns a structured JSON risk assessment (defense). Both are deliberately simplified, conceptual snippets (the book
notes that a production Alpha Digest would sit on a RAG pipeline), and both call the OpenAI Responses API, so you need
the `openai` package and an `OPENAI_API_KEY`.

## Setup (not a book listing)
The listings create `openai.OpenAI()`, which reads `OPENAI_API_KEY` from the environment. If you keep the key in a
`.env` file as described in appendix A.4, run the next cell first: `load_dotenv()` searches this folder and its
parents, so a `.env` file in the repository root is found. Running the listings sends real (billed) requests to the
OpenAI API.

In [ ]:
# Setup helper (not part of the printed listings): load OPENAI_API_KEY from a .env file
# in this folder or any parent folder (see appendix A.4).
import os
from dotenv import load_dotenv

load_dotenv()
print("OPENAI_API_KEY is set." if os.getenv("OPENAI_API_KEY")
      else "OPENAI_API_KEY is not set: openai.OpenAI() in the listings below will raise an error.")

### Listing 2.1 — Conceptual snippet for Alpha Digest summarization
*Section 2.3.1, The offense play: Creating a new revenue stream (pp. 23–25)*

The offense strategy: turn the posts of investors a user follows into a premium digest that extracts the investment
thesis, price targets, and overall sentiment (`gpt-4o-mini`, low temperature).

In [ ]:
import openai
from typing import List, Dict, Optional

# Best practice: Initialize the client. The API key is loaded automatically
# from the OPENAI_API_KEY environment variable.
client = openai.OpenAI()


def generate_alpha_digest(posts: List[str], user_profile: Dict) -> Optional[str]:
    """
    Generates a personalized summary of investment posts using the Responses API.
    Includes basic input validation and error handling for robustness.
    """
    # 1. Input Validation: Ensure there are posts to analyze.
    if not posts:
        return "No posts provided for analysis."

    # Safely read the user's name, with a neutral fallback if it is missing.
    user_name = user_profile.get("name", "the user")

    # Format the posts for the prompt, taking up to 3 for this example.
    posts_text = "\n".join([f'- "{post}"' for post in posts[:3]])

    prompt_content = f"""
    You are an expert financial analyst. Based on the following posts from investors
    that {user_name} follows, please provide a concise 'Alpha Digest'.
    Extract the key investment thesis, any mentioned price targets, and the overall sentiment.

    Posts:
    {posts_text}

    Your Premium Digest:
    """

    try:
        # 2. Modern API Call: Use the Responses API.
        response = client.responses.create(
            model="gpt-4o-mini",  # A current, cost-effective model
            input=prompt_content,
            max_output_tokens=200,
            temperature=0.2,
        )
        return response.output_text.strip()
    # 3. Error Handling: Catch OpenAI API errors specifically.
    except openai.APIError as e:
        # In a real application, you would log this error.
        print(f"An OpenAI API error occurred: {e}")
        return "Analysis is temporarily unavailable."


# --- Example Usage ---
sample_posts = [
    "I'm still bullish on $TECHCO. Their recent acquisition in the AI space is a game-changer. PT $150.",
    "Seeing a lot of institutional buying in $ENERGIZE. The move to green energy is a long-term tailwind. Accumulating.",
    "Warning on $RETAILCORP. Supply chain issues are worsening, and their margins will get squeezed. I'm staying away.",
]

user = {"name": "Jane Doe"}

digest = generate_alpha_digest(sample_posts, user)
print(digest)

### Listing 2.2 — Conceptual snippet for an internal moderation tool
*Section 2.3.2, The defense play: Boosting productivity and retention (pp. 25–26)*

The defense strategy: the same LLM summarizes a flagged post for a human moderator and returns a JSON object with a
one-sentence summary and a `risk_level`, using the Responses API's JSON mode
(`text={"format": {"type": "json_object"}}`).

In [ ]:
import openai
from typing import Dict, Optional
import json

client = openai.OpenAI()

def summarize_post_for_moderator(post_text: str) -> Optional[Dict]:
    """
    Summarizes a post for a human moderator, returning a structured JSON object.
    Includes validation and error handling.
    """
    # 1. Input Validation: Don't process empty posts.
    if not post_text.strip():
        return {"error": "Post text is empty."}

    prompt_content = f"""
    You are a moderation assistant for the AlphaStream financial community.
    Analyze the following post for potential policy violations (e.g., pumping, personal attacks).
    Provide a one-sentence summary and a "risk_level" (Low, Medium, High) with a brief justification.
    Your response MUST be in a valid JSON format.

    Post:
    "{post_text}"

    Your Analysis (JSON):
    """

    try:
        # 2. Modern API Call: Use the Responses API with JSON mode.
        response = client.responses.create(
            model="gpt-4o-mini",
            input=prompt_content,
            max_output_tokens=300,
            temperature=0.0,
            text={"format": {"type": "json_object"}},
        )

        # 3. JSON Parsing and Validation
        analysis_str = response.output_text.strip()
        return json.loads(analysis_str)

    # 4. Error Handling for both API and JSON parsing issues.
    except json.JSONDecodeError:
        return {"error": "Failed to parse model output as JSON.", "raw_output": analysis_str}
    except openai.APIError as e:
        print(f"An OpenAI API error occurred: {e}")
        return {"error": "Moderation analysis is temporarily unavailable."}

# --- Example Usage ---
problematic_post = "Everyone BUY $ROCKET NOW!!! This is guaranteed to go to the moon, I have inside info. Anyone who disagrees is an idiot."

moderation_summary = summarize_post_for_moderator(problematic_post)
print(moderation_summary)